In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.models import Sequential, load_model
from keras.layers import Dense, Activation, Dropout, TimeDistributed, LSTM
from keras.optimizers import RMSprop
from keras.callbacks import Callback, ModelCheckpoint, EarlyStopping
import warnings
warnings.filterwarnings("ignore", category = UserWarning, module="keras")

In [5]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

GPU Memory Growth Enabled


In [6]:
training_file = 'warpeace_input.txt'
raw_text = open(training_file, 'r').read()
raw_text = raw_text.lower()
raw_text[:100]

'"well, prince, so genoa and lucca are now just family estates of the\nbuonapartes. but i warn you, if'

In [7]:
n_chars = len(raw_text)
print(f'Total characters: {n_chars}')

Total characters: 3196213


In [8]:
chars = sorted(list(set(raw_text)))
n_vocab = len(chars)
print('Total vocabulary (unique characters) : {}'.format(n_vocab))
print(chars)

Total vocabulary (unique characters) : 56
['\n', ' ', '!', '"', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'ä', 'é', 'ê']


In [9]:
index_to_char = dict((i, c) for i , c in enumerate(chars))
char_to_index = dict((c , i) for i , c in enumerate(chars))
print(char_to_index) 

{'\n': 0, ' ': 1, '!': 2, '"': 3, "'": 4, '(': 5, ')': 6, '*': 7, ',': 8, '-': 9, '.': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, ';': 23, '=': 24, '?': 25, 'a': 26, 'b': 27, 'c': 28, 'd': 29, 'e': 30, 'f': 31, 'g': 32, 'h': 33, 'i': 34, 'j': 35, 'k': 36, 'l': 37, 'm': 38, 'n': 39, 'o': 40, 'p': 41, 'q': 42, 'r': 43, 's': 44, 't': 45, 'u': 46, 'v': 47, 'w': 48, 'x': 49, 'y': 50, 'z': 51, 'à': 52, 'ä': 53, 'é': 54, 'ê': 55}


In [10]:
seq_length = 160
n_seq = int(n_chars / seq_length)
n_seq

19976

In [11]:
X = np.zeros((n_seq, seq_length, n_vocab))
Y = np.zeros((n_seq, seq_length, n_vocab))

In [12]:
for i in range(n_seq):
    x_sequence = raw_text[i * seq_length : (i + 1) * seq_length]
    x_sequence_ohe = np.zeros((seq_length, n_vocab))
    for j in range(seq_length):
        char = x_sequence[j]
        index = char_to_index[char]
        x_sequence_ohe[j][index] = 1.
    X[i] = x_sequence_ohe
    y_sequence = raw_text[i * seq_length + 1 : (i + 1) * seq_length + 1]

    y_sequence_ohe = np.zeros((seq_length, n_vocab))
    for j in range(seq_length):
        char = y_sequence[j]
        index = char_to_index[char]
        y_sequence_ohe[j][index] = 1.
    Y[i] = y_sequence_ohe

In [13]:
batch_size = 100
n_layer = 2
hidden_units = 800
n_epoch = 300
dropout = 0.4

In [ ]:
model = Sequential()
model.add(LSTM(hidden_units, input_shape = (None, n_vocab), return_sequences = True))
model.add(Dropout(dropout))

for i in range(n_layer - 1):
    model.add(LSTM(hidden_units, return_sequences=True))
    model.add(Dropout(dropout))

model.add(TimeDistributed(Dense(n_vocab)))
model.add(Activation('softmax'))

optimizer = RMSprop(learning_rate = 0.001, rho = 0.9, epsilon = 1e-08, decay = 0.0)

model.compile(loss = "categorical_crossentropy", optimizer = optimizer)
model.summary()

I0000 00:00:1789131940.938488  970674 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4130 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [ ]:
filepath = "lstm_weights_epoch_{epoch:03d}_loss_{loss:.4f}.keras"
checkpoint = ModelCheckpoint(filepath, monitor = 'loss', verbose = 1, save_best_only = True, mode = 'min')
early_stop = EarlyStopping(monitor = 'loss', min_delta = 0, patience = 50, verbose = 1, mode = 'min')

NameError: name 'ModelCheckpoint' is not defined

In [ ]:
def generate_text(model, gen_length, n_vocab, index_to_char):
    index = np.random.randint(n_vocab)
    y_char = [index_to_char[index]]
    X = np.zeros((1, gen_length, n_vocab))
    
    for i in range(gen_length):
        X[0, i, index] = 1.
        indices = np.argmax(model.predict(X[:, max(0, i - 99):i + 1, :])[0], 1)
        index = indices[-1]
        y_char.append(index_to_char[index])
        
    return ('').join(y_char)


In [ ]:
class ResultChecker(Callback):
    def __init__(self, N, gen_length):
        self.N = N
        self.gen_length = gen_length
        
    def on_epoch_end(self, epoch, logs={}):
        if epoch % self.N == 0:
            result = generate_text(self.model, self.gen_length, n_vocab, index_to_char)
            print('\nMy War and Peace:\n' + result)


In [ ]:
model.fit(X, Y, batch_size = batch_size, verbose = 1, epochs = n_epoch, 
    callbacks = [ResultChecker(10, 200), checkpoint, early_stop])

In [ ]:
model = load_model('weights_epoch_031_loss_0.0000.keras') 
print("Model loaded successfully!")

generated_result = generate_text(model, gen_length=500, n_vocab=n_vocab, index_to_char=index_to_char)

print("\n--- Generated Text from Loaded Model: ---\n")
print(generated_result)